In [ ]:
!pip install langchain langgraph pandas langchain-openrouter fastmcp


In [2]:
import os
# ====== PUT YOUR KEYS HERE ======
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"
os.environ["OPENROUTER_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_KEY"

In [5]:
from langchain.chat_models import init_chat_model
def get_llm(provider):
  if provider=="openrouter":
    llm = init_chat_model(
        "openai/gpt-4o-mini",
        model_provider="openrouter"
    )
  elif provider=="chatgpt":
    llm = init_chat_model("gpt-4o-mini", model_provider="openai")
  elif provider=="gemini":
    llm = init_chat_model(
      "gemini-1.5-pro",
      model_provider="google_genai"
    )
  else:
    raise Exception("Invalid provider")
  return llm
# --- OPTION 1: OpenAI ---
model="openrouter" #chatgpt, gemini
llm = get_llm(provider=model)


In [14]:
import pandas as pd
from langchain.tools import tool
#from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent

@tool
def clean_csv(file_path: str) -> str:
    """
    Load a CSV and perform basic cleaning:
    - drop duplicates
    - fill NA with median (numeric)
    - return summary
    """
    df = pd.read_csv(file_path)

    df = df.drop_duplicates()

    for col in df.select_dtypes(include="number").columns:
        df[col] = df[col].fillna(df[col].median())

    return f"Cleaned DataFrame shape: {df.shape}\nColumns: {list(df.columns)}"


tools = [clean_csv]

agent = create_agent(
    model=llm,
    tools=tools,
)

In [7]:
%%bash
#rm -r gdpymes/
git clone https://github.com/bsaldivaremc2/gdpymes.git

Cloning into 'gdpymes'...


In [8]:
%%bash
unzip /content/gdpymes/datasets/wine+quality.zip

Archive:  /content/gdpymes/datasets/wine+quality.zip
  inflating: winequality-red.csv     
  inflating: winequality-white.csv   
  inflating: winequality.names       


In [11]:
csv_path = "/content/winequality-red.csv"

In [12]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Clean the CSV at "+csv_path}]
})


In [13]:
result

{'messages': [HumanMessage(content='Clean the CSV at /content/winequality-red.csv', additional_kwargs={}, response_metadata={}, id='86a64acc-51fc-42c7-9551-8ff2dc50b8de'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1774192846-a3pg7E3iVGrprqYTEnht', 'created': 1774192846, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'system_fingerprint': 'fp_ca3e7d71bf'}, id='lc_run--019d1622-64ef-79e2-b6a0-8240350afe06-0', tool_calls=[{'name': 'clean_csv', 'args': {'file_path': '/content/winequality-red.csv'}, 'id': 'call_HcXq41TLwQEsKGHL9pvsaUJK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 21, 'total_tokens': 95, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}),
  ToolMessage(content='Cleaned DataFrame shape: (1359, 1)\nColumns: [\'fixed acidity;"volatile aci

In [ ]:
import importlib
import sys

SUPPORT_DIRECTORY = "gdpymes/march2026/"
sys.path.append(SUPPORT_DIRECTORY)